## **1. Setup**

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.data_pipeline import DatabaseManager, DataLoader, Preprocessor
from src.events import SimpleEventDetector

## **2. Load Data**

In [2]:
db_path = Path("../NordPoool/data/thesis_database.db")

db = DatabaseManager(db_path)
loader = DataLoader(db)
pre = Preprocessor()

df_prices = loader.load_prices()

## **3. Build DateTime**

In [3]:
import pandas as pd

df_prices["datetime"] = pd.to_datetime(df_prices["delivery_day"]) + pd.to_timedelta(df_prices["hour"], unit="h")

## **4. Detect Price Events**

In [4]:
event_detector = SimpleEventDetector()

df_price_events = event_detector.detect_price_events(df_prices)

## **5. Check Results**

In [5]:
df_price_events[[
    "datetime",
    "zone_id",
    "price_value",
    "high_price",
    "extreme_price",
    "low_price",
    "price_spike"
]].head()

,datetime,zone_id,price_value,high_price,extreme_price,low_price,price_spike
0,2020-01-01 00:00:00,1,28.78,False,False,False,False
20,2020-01-01 01:00:00,1,28.45,False,False,False,False
40,2020-01-01 02:00:00,1,27.90,False,False,False,False
60,2020-01-01 03:00:00,1,27.52,False,False,False,False
80,2020-01-01 04:00:00,1,27.54,False,False,False,False


## **6. Count Events**

In [6]:
event_counts = df_price_events[[
    "high_price",
    "extreme_price",
    "low_price",
    "price_spike"
]].sum().reset_index()

event_counts.columns = ["event_type", "count"]

event_counts

,event_type,count
0,high_price,100790
1,extreme_price,50385
2,low_price,100714
3,price_spike,100767
